# GlobalLearn Course Sales and Advertising Analysis
## Matplotlib Project Using an International Sample Dataset

This is an educational project about a **fictional** online learning company serving India, the United States, the United Kingdom, Canada, and Australia. The numbers are simulated for practice and are not actual BYJU’S or any company records. The sample covers 2024 course categories, advertising channels, impressions, clicks, enrollments, ad spend, and estimated revenue. Currency values are converted to illustrative USD estimates for comparisons.

In [ ]:
!pip -q install pandas matplotlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files

## 1. Create and download the sample dataset
Run this cell to create the same CSV supplied with this notebook. It is synthetic sample data.

In [ ]:
countries = {
    "India": ("INR", 83, 1.0),
    "United States": ("USD", 1, 1.45),
    "United Kingdom": ("GBP", 0.79, 1.25),
    "Canada": ("CAD", 1.36, 1.15),
    "Australia": ("AUD", 1.52, 1.10)
}
courses = [("Mathematics",100), ("Science",120), ("Coding",150), ("English",80)]
channels = ["Search Ads", "Social Media", "Video Ads"]
month_factor = [0.88,0.90,0.95,0.98,1.02,1.08,1.12,1.10,1.05,1.00,1.08,1.20]
records = []
for mi in range(12):
    for ci, (country, (currency, per_usd, market)) in enumerate(countries.items()):
        for ki, (course, base_usd) in enumerate(courses):
            for ai, channel in enumerate(channels):
                spend_usd = round((280 + ci*65 + ki*30 + ai*45) * (0.82 + 0.035*mi), 2)
                impressions = int(spend_usd * [950,1250,820][ai] * (1 + 0.015*mi))
                clicks = int(impressions * [0.045,0.055,0.038][ai])
                enrollments = int(clicks * [0.075,0.058,0.045][ai] * month_factor[mi] * market)
                price_usd = base_usd * (0.68 if country == "India" else (0.90 if country == "Canada" else 1.0))
                revenue_usd = round(enrollments * price_usd, 2)
                records.append({
                    "Month": f"2024-{mi+1:02d}", "Country": country,
                    "Course_Category": course, "Advertising_Channel": channel,
                    "Currency": currency, "Ad_Spend_Local": round(spend_usd*per_usd,2),
                    "Impressions": impressions, "Clicks": clicks, "Enrollments": enrollments,
                    "Course_Price_Local": round(price_usd*per_usd,2),
                    "Revenue_Local": round(revenue_usd*per_usd,2),
                    "Ad_Spend_USD_Estimate": spend_usd, "Revenue_USD_Estimate": revenue_usd,
                    "Note": "SIMULATED EDUCATIONAL DATA - not actual company records"
                })
df = pd.DataFrame(records)
df.to_csv("GlobalLearn_Simulated_Course_Ads_Sales.csv", index=False)
display(df.head())
print(f"Rows: {len(df)} | Countries: {df.Country.nunique()} | Months: {df.Month.nunique()}")
files.download("GlobalLearn_Simulated_Course_Ads_Sales.csv")

## 2. Summary
Inspect the sample before plotting.

In [ ]:
display(df[["Ad_Spend_USD_Estimate","Impressions","Clicks","Enrollments","Revenue_USD_Estimate"]].describe().round(2))
country_summary = df.groupby("Country").agg(Enrollments=("Enrollments","sum"), Ad_Spend_USD=("Ad_Spend_USD_Estimate","sum"), Revenue_USD=("Revenue_USD_Estimate","sum"))
country_summary["ROAS"] = country_summary["Revenue_USD"] / country_summary["Ad_Spend_USD"]
display(country_summary.round(2))

## 3. Line graph: monthly course enrollments
Shows the trend by country using `plot`, markers, marker colors, line formatting, title/axis/tick properties, grid, and legend.

In [ ]:
df["Month_Date"] = pd.to_datetime(df["Month"] + "-01")
monthly_country = df.groupby(["Month_Date","Country"])["Enrollments"].sum().unstack()
plt.figure(figsize=(11,6), dpi=150, facecolor="#f7f9fc")
markers = ["o","s","^","D","*"]
for i, country in enumerate(monthly_country.columns):
    plt.plot(monthly_country.index, monthly_country[country], linewidth=2, linestyle="-", marker=markers[i], markersize=6, markerfacecolor="white", markeredgecolor="black", alpha=0.85, label=country)
plt.title("Monthly Course Enrollments by Country", fontsize=16, fontweight="bold", fontstyle="italic", color="navy", loc="center", pad=15, family="sans-serif", alpha=0.95)
plt.xlabel("Month", fontsize=13, fontweight="bold", color="darkred", labelpad=10, family="sans-serif")
plt.ylabel("Enrollments", fontsize=13, fontweight="bold", color="darkgreen", labelpad=10)
plt.xticks(monthly_country.index, monthly_country.index.strftime("%b"), fontsize=10, rotation=35, color="blue", fontweight="medium")
plt.yticks(fontsize=10, color="black", fontweight="medium")
plt.grid(True, color="gray", linestyle=":", alpha=0.6, linewidth=0.8)
plt.legend(loc="best", fontsize=10, frameon=True, shadow=False)
plt.tight_layout()
plt.show()

## 4. Bar graph: revenue by country
Revenue is displayed as an illustrative USD estimate so countries can be compared. Error bars show month-to-month standard deviation. Includes `bottom`, `bar_label`, hatch, width, alignment, border, transparency, and error bar options.

In [ ]:
country_rev = df.groupby("Country")["Revenue_USD_Estimate"].sum().sort_values(ascending=False)
country_month = df.groupby(["Country","Month"])["Revenue_USD_Estimate"].sum()
errors = country_month.groupby("Country").std().reindex(country_rev.index).fillna(0)
plt.figure(figsize=(10,6), dpi=150, facecolor="white")
bars = plt.bar(country_rev.index, country_rev.values, color="skyblue", width=0.62, edgecolor="black", linewidth=1.5, alpha=0.9, align="center", bottom=0, label="Estimated revenue", hatch="//", yerr=errors.values, capsize=5)
plt.title("Estimated Course Revenue by Country", fontsize=16, fontweight="bold", color="purple", loc="center", pad=15, family="serif", alpha=0.95)
plt.xlabel("Country", fontsize=13, color="red", labelpad=8)
plt.ylabel("Estimated revenue (USD)", fontsize=13, color="darkgreen", labelpad=8)
plt.xticks(fontsize=10, rotation=20, color="blue", fontweight="bold")
plt.yticks(fontsize=10, color="black")
plt.bar_label(bars, labels=[f"${v:,.0f}" for v in country_rev.values], fontsize=9, padding=3)
plt.grid(axis="y", color="gray", linestyle="--", alpha=0.5, linewidth=0.8)
plt.legend(loc="upper right", fontsize=10, frameon=True, shadow=True)
plt.tight_layout()
plt.show()

## 5. Horizontal bar graph: advertising spend by channel
Demonstrates `barh`, custom tick labels, and its `left` baseline option.

In [ ]:
channel_spend = df.groupby("Advertising_Channel")["Ad_Spend_USD_Estimate"].sum().sort_values()
plt.figure(figsize=(9,5), dpi=150, facecolor="white")
plt.barh(channel_spend.index, channel_spend.values, height=0.65, left=0, color="teal", edgecolor="black", linewidth=1, alpha=0.85, label="Estimated ad spend")
plt.title("Advertising Spend by Channel", fontsize=15, fontweight="bold", color="navy", pad=12)
plt.xlabel("Estimated ad spend (USD)", fontsize=12, color="darkred", labelpad=8)
plt.ylabel("Advertising channel", fontsize=12, color="darkgreen", labelpad=8)
plt.xticks(fontsize=10, color="black")
plt.yticks(fontsize=10, color="blue", fontweight="bold")
plt.grid(axis="x", color="gray", linestyle=":", alpha=0.6)
plt.legend(loc="best", fontsize=10, frameon=True)
plt.tight_layout()
plt.show()

## 6. Pie chart: enrollments by course category
Includes colors, percentage labels, start angle, exploded slice, shadow, wedge borders, text formatting, direction, and label distances.

In [ ]:
course_sales = df.groupby("Course_Category")["Enrollments"].sum()
explode = [0.08 if x == course_sales.max() else 0 for x in course_sales]
plt.figure(figsize=(8,8), dpi=150, facecolor="#f7f9fc")
plt.pie(course_sales.values, labels=course_sales.index, colors=["tomato","skyblue","lightgreen","orange"], autopct="%1.1f%%", startangle=90, explode=explode, shadow=True, wedgeprops={"edgecolor":"black","linewidth":1}, textprops={"fontsize":11,"color":"black"}, counterclock=True, pctdistance=0.7, labeldistance=1.1)
plt.title("Share of Enrollments by Course Category", fontsize=15, fontweight="bold", color="navy", pad=20)
plt.legend(loc="best", fontsize=9, frameon=True)
plt.tight_layout()
plt.show()

## 7. Scatter plot: ad spend and estimated revenue
Color represents enrollments; this helps inspect the relationship between marketing cost and course sales.

In [ ]:
plt.figure(figsize=(9,6), dpi=150, facecolor="white")
sc = plt.scatter(df["Ad_Spend_USD_Estimate"], df["Revenue_USD_Estimate"], c=df["Enrollments"], cmap="viridis", alpha=0.65, edgecolors="black", linewidths=0.3)
cb = plt.colorbar(sc); cb.set_label("Enrollments")
plt.title("Estimated Revenue vs Advertising Spend", fontsize=15, fontweight="bold", color="navy", pad=12)
plt.xlabel("Estimated ad spend (USD)", fontsize=12, color="darkred", labelpad=8)
plt.ylabel("Estimated revenue (USD)", fontsize=12, color="darkgreen", labelpad=8)
plt.xticks(fontsize=10, rotation=0, color="blue"); plt.yticks(fontsize=10, color="black")
plt.grid(True, color="gray", linestyle="--", alpha=0.45)
plt.tight_layout(); plt.show()

## 8. Save a chart
The image is saved at high resolution in the Colab session.

In [ ]:
plt.figure(figsize=(10,5), dpi=150, facecolor="white")
plt.plot(monthly_country.index, monthly_country.sum(axis=1), color="crimson", linewidth=2, marker="o", markersize=5, label="All sample markets")
plt.title("Monthly Enrollments Across All Countries", fontsize=15, fontweight="bold", color="navy", pad=12)
plt.xlabel("Month", fontsize=12, color="darkred", labelpad=8); plt.ylabel("Enrollments", fontsize=12, color="darkgreen", labelpad=8)
plt.xticks(monthly_country.index, monthly_country.index.strftime("%b"), fontsize=9, rotation=40, color="blue"); plt.yticks(fontsize=10, color="black")
plt.grid(True, color="gray", linestyle="--", alpha=0.5); plt.legend(loc="best", fontsize=10, frameon=True)
plt.tight_layout()
plt.savefig("course_enrollments_by_month.png", dpi=300, format="png", bbox_inches="tight", transparent=False)
plt.show()
print("Saved course_enrollments_by_month.png")

## 9. Area chart: monthly revenue by advertising channel
An area chart makes the changing contribution of each channel visible over time.

In [ ]:
channel_month = df.groupby(["Month","Advertising_Channel"])["Revenue_USD_Estimate"].sum().unstack().reindex(range(1,13))
plt.figure(figsize=(10,5), dpi=140, facecolor="white")
plt.stackplot(range(1,13), *[channel_month[c].values for c in channel_month.columns], labels=channel_month.columns, alpha=0.78)
plt.title("Monthly Estimated Revenue by Ad Channel", fontsize=15, fontweight="bold")
plt.xlabel("Month"); plt.ylabel("Estimated revenue (USD)")
plt.xticks(range(1,13), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], rotation=35)
plt.grid(axis="y", alpha=0.35); plt.legend(loc="upper left", fontsize=9)
plt.tight_layout(); plt.show()

## 10. Stacked bar chart: enrollments by country and channel

In [ ]:
country_channel = df.groupby(["Country","Advertising_Channel"])["Enrollments"].sum().unstack(fill_value=0)
plt.figure(figsize=(10,5), dpi=140, facecolor="white")
bottom = [0] * len(country_channel)
for channel in country_channel.columns:
    plt.bar(country_channel.index, country_channel[channel], bottom=bottom, label=channel, edgecolor="white")
    bottom = [a+b for a,b in zip(bottom, country_channel[channel].values)]
plt.title("Enrollments by Country and Ad Channel", fontsize=15, fontweight="bold")
plt.xlabel("Country"); plt.ylabel("Enrollments")
plt.xticks(rotation=20); plt.grid(axis="y", alpha=0.35); plt.legend()
plt.tight_layout(); plt.show()

## 11. Histogram: distribution of enrollment counts per campaign row

In [ ]:
plt.figure(figsize=(9,5), dpi=140, facecolor="white")
plt.hist(df["Enrollments"], bins=20, color="cornflowerblue", edgecolor="black", alpha=0.8)
plt.axvline(df["Enrollments"].mean(), color="crimson", linestyle="--", linewidth=2, label=f"Mean = {df.Enrollments.mean():.1f}")
plt.title("Distribution of Sample Enrollments", fontsize=15, fontweight="bold")
plt.xlabel("Enrollments per country-course-channel-month record"); plt.ylabel("Number of records")
plt.grid(axis="y", alpha=0.35); plt.legend(); plt.tight_layout(); plt.show()

## 12. Box plot: estimated revenue by country
These are simulated comparisons; the boxes summarize the monthly sample records.

In [ ]:
countries_order = list(countries.keys())
box_data = [df.loc[df.Country==c, "Revenue_USD_Estimate"].values for c in countries_order]
plt.figure(figsize=(10,5), dpi=140, facecolor="white")
bp = plt.boxplot(box_data, tick_labels=countries_order, patch_artist=True, showmeans=True)
for patch, color in zip(bp["boxes"], ["skyblue","salmon","lightgreen","gold","plum"]): patch.set_facecolor(color)
plt.title("Estimated Revenue Distribution by Country", fontsize=15, fontweight="bold")
plt.xlabel("Country"); plt.ylabel("Estimated revenue (USD)")
plt.xticks(rotation=20); plt.grid(axis="y", alpha=0.35); plt.tight_layout(); plt.show()

## 13. Violin plot: revenue distribution by country

In [ ]:
plt.figure(figsize=(10,5), dpi=140, facecolor="white")
vp = plt.violinplot(box_data, showmeans=True, showmedians=True, showextrema=True)
for body, color in zip(vp["bodies"], ["skyblue","salmon","lightgreen","gold","plum"]): body.set_facecolor(color); body.set_alpha(0.75)
plt.title("Estimated Revenue Density by Country", fontsize=15, fontweight="bold")
plt.xlabel("Country"); plt.ylabel("Estimated revenue (USD)")
plt.xticks(range(1,len(countries_order)+1), countries_order, rotation=20)
plt.grid(axis="y", alpha=0.35); plt.tight_layout(); plt.show()

## 14. Heatmap: monthly enrollments by country

In [ ]:
heat = monthly_country.T.values
plt.figure(figsize=(11,4.5), dpi=140, facecolor="white")
im = plt.imshow(heat, aspect="auto", cmap="YlGnBu", interpolation="nearest")
plt.colorbar(im, label="Enrollments")
plt.title("Enrollment Heatmap: Country by Month", fontsize=15, fontweight="bold")
plt.xlabel("Month"); plt.ylabel("Country")
plt.xticks(range(12), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
plt.yticks(range(len(monthly_country.columns)), monthly_country.columns)
plt.tight_layout(); plt.show()

## 15. Donut chart: course enrollment share

In [ ]:
plt.figure(figsize=(7,7), dpi=140, facecolor="white")
plt.pie(course_sales.values, labels=course_sales.index, autopct="%1.1f%%", startangle=90, wedgeprops={"width":0.42,"edgecolor":"white"}, pctdistance=0.78)
plt.title("Course Enrollment Share (Donut Chart)", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()

## 16. Step chart: monthly advertising spend

In [ ]:
monthly_spend = df.groupby("Month")["Ad_Spend_USD_Estimate"].sum().reindex(range(1,13))
plt.figure(figsize=(9,4.5), dpi=140, facecolor="white")
plt.step(monthly_spend.index, monthly_spend.values, where="mid", color="darkorange", linewidth=2, label="Estimated ad spend")
plt.plot(monthly_spend.index, monthly_spend.values, marker="o", color="darkorange")
plt.title("Monthly Estimated Advertising Spend", fontsize=15, fontweight="bold")
plt.xlabel("Month"); plt.ylabel("Estimated ad spend (USD)")
plt.xticks(range(1,13)); plt.grid(True, alpha=0.35); plt.legend()
plt.tight_layout(); plt.show()

## 17. Error-bar chart: monthly enrollments and variation across countries

In [ ]:
monthly_mean = monthly_country.mean(axis=1)
monthly_std = monthly_country.std(axis=1)
plt.figure(figsize=(9,4.5), dpi=140, facecolor="white")
plt.errorbar(range(1,13), monthly_mean.values, yerr=monthly_std.values, fmt="o-", color="teal", ecolor="gray", capsize=4, linewidth=1.8, label="Mean across countries ± standard deviation")
plt.title("Monthly Enrollments with Country Variation", fontsize=15, fontweight="bold")
plt.xlabel("Month"); plt.ylabel("Enrollments")
plt.xticks(range(1,13)); plt.grid(True, alpha=0.35); plt.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 18. Hexbin chart: advertising spend and estimated revenue

In [ ]:
plt.figure(figsize=(8,5), dpi=140, facecolor="white")
hb = plt.hexbin(df["Ad_Spend_USD_Estimate"], df["Revenue_USD_Estimate"], gridsize=25, cmap="magma", mincnt=1)
plt.colorbar(hb, label="Number of sample records")
plt.title("Density of Ad Spend and Estimated Revenue", fontsize=15, fontweight="bold")
plt.xlabel("Estimated ad spend (USD)"); plt.ylabel("Estimated revenue (USD)")
plt.grid(alpha=0.2); plt.tight_layout(); plt.show()

## 19. Stem plot: course category enrollments

In [ ]:
course_totals = df.groupby("Course_Category")["Enrollments"].sum()
plt.figure(figsize=(8,4.5), dpi=140, facecolor="white")
plt.stem(course_totals.index, course_totals.values, linefmt="C0-", markerfmt="C1o", basefmt="k-")
plt.title("Total Enrollments by Course Category", fontsize=15, fontweight="bold")
plt.xlabel("Course category"); plt.ylabel("Enrollments")
plt.xticks(rotation=15); plt.grid(axis="y", alpha=0.35); plt.tight_layout(); plt.show()

## Matplotlib functions and chart types covered
The notebook demonstrates line, vertical and horizontal bar, stacked bar, area, pie, donut, scatter, histogram, box, violin, heatmap, step, error-bar, hexbin, and stem charts. It uses `plt.figure`, `plt.plot`, `plt.step`, `plt.bar`, `plt.barh`, `plt.stackplot`, `plt.pie`, `plt.scatter`, `plt.hist`, `plt.boxplot`, `plt.violinplot`, `plt.imshow`, `plt.errorbar`, `plt.hexbin`, `plt.stem`, `plt.colorbar`, `plt.title`, `plt.xlabel`, `plt.ylabel`, `plt.xticks`, `plt.yticks`, `plt.grid`, `plt.legend`, `plt.bar_label`, `plt.tight_layout`, `plt.savefig`, and `plt.show`, along with the formatting properties listed in the assignment. Matplotlib has many additional specialized plot types; this notebook covers a broad set suited to this dataset. The business data is simulated educational data, not actual BYJU’S records.